# 01 — Data Preparation
Load RAGTruth, generate answers, extract 17 signals, label with the ensemble judge, and cache to CSV.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()   # reads GROQ_API_KEY_1/2/3 from .env


## Imports

In [ ]:
import pandas as pd
from tqdm import tqdm
from signalrag import SignalExtractor, EnsembleJudge, load_ragtruth
from signalrag.config import N_TRAIN, N_TEST, TRAIN_CSV, TEST_CSV, FEATURE_COLS, LABEL_COL


## Load models

In [ ]:
# Check cache BEFORE loading heavy models (Issue 4 fix)
import os
os.makedirs("data", exist_ok=True)

if os.path.exists(TRAIN_CSV):
    print(f"[CACHE] {TRAIN_CSV} already exists — skipping model load and pipeline")
    extractor = None
    judge     = None
else:
    extractor = SignalExtractor(load_llm=True)
    judge     = EnsembleJudge()
    judge.verify()


## Run pipeline

In [ ]:
def run_pipeline(records, desc):
    rows = []
    judge_failures = 0
    for rec in tqdm(records, desc=desc):
        query, passages, context = rec["query"], rec["passages"], rec["context"]
        gen_text, signals = extractor.extract_all_signals(query, passages, context)
        label = judge.judge(gen_text, context)
        if label == -1:
            judge_failures += 1
        rows.append({
            "query": query,
            "generated_answer": gen_text,
            "context": context[:600],
            **signals,
            LABEL_COL: label,
        })
    df = pd.DataFrame(rows)
    failed = (df[LABEL_COL] == -1).sum()
    if failed:
        print(f"  Dropping {failed} rows where all judges failed")
        df = df[df[LABEL_COL] != -1].reset_index(drop=True)
    print(f"  Judge failures: {judge_failures}")
    return df


In [ ]:
if extractor is not None:
    records = load_ragtruth(N_TRAIN, split="train")
    df = run_pipeline(records, desc="train")
    df.to_csv(TRAIN_CSV, index=False)
    dist = df[LABEL_COL].value_counts().to_dict()
    print(f"  train: {dist.get(1,0)} faithful / {dist.get(0,0)} hallucinated -> {TRAIN_CSV}")
